# Pichler et al. (2022) Dynamic Input-Output Model (PichlerEtAl2022DIO)

This model implements the dynamic disequilibrium input-output model introduced by
Pichler, Pangallo, del Rio-Chanona, Lafond & Farmer (2022),
*Forecasting the propagation of pandemic shocks with a dynamic input-output model*,
Journal of Economic Dynamics and Control, 144, 104527.

The model simulates daily production, consumption and employment dynamics
across 55 industries (WIOD classification) subject to pandemic-induced
supply and demand shocks.  Key features include:

- **Partially Binding Leontief (PBL) production functions** that distinguish
  critical from non-critical intermediate inputs.
- **Industry-specific inventories** initialised from ONS survey data.
- **Muellbauer consumption function** incorporating fear-of-infection and
  permanent-income effects.
- **Sluggish labour adjustment** with asymmetric hiring/firing speeds.

## Module Contents

As with all `MacroStat` models, PichlerEtAl2022DIO is divided into
Variables, Parameters, Scenarios, and the Behavior (model initialisation
and steps).  The module-level documentation can be found in:

```{eval-rst}
.. toctree::
    :maxdepth: 2

    Variables <variables.rst>
    Parameters <parameters.rst>
    Equations <equations.rst>
    Scenarios <scenarios.rst>
```

For the full model API please check the [API Reference](../../api_reference).

## Model Overview

A time step corresponds to one calendar day.  There are $N = 55$ industries
and one representative household.  The economy initially rests in a
steady state until it experiences exogenous pandemic shocks.  Every day
the model executes the following steps:

1. **Labour adjustment** -- firms hire or fire workers depending on
   whether their workforce was insufficient or redundant.
2. **Demand** -- households decide consumption; industries place
   intermediate-goods orders.
3. **Production** -- industries produce subject to labour capacity,
   input availability, and demand constraints.
4. **Rationing** -- if output < demand, production is distributed
   pro rata across customers.
5. **Inventories & accounting** -- inventory levels are updated;
   profits and savings are computed.

### Key Equations

**Output identity (Eq. 1)**

$$
x_{i,t} = \sum_{j=1}^{N} Z_{ij,t} + c_{i,t} + f_{i,t}
$$

**Intermediate demand (Eq. 4)**

$$
O_{ji,t} = A_{ji}\,d_{i,t-1} + \frac{1}{\tau}\left(n_i Z_{ji,0} - S_{ji,t-1}\right)
$$

**Aggregate consumption (Eq. 6)**

$$
\tilde{c}^d_t = (1 - \tilde{\epsilon}^D_t)\,
  \exp\!\left(
    \rho \log \tilde{c}^d_{t-1}
    + \frac{1-\rho}{2}\log(m\tilde{l}_t)
    + \frac{1-\rho}{2}\log(m\tilde{l}^p_t)
  \right)
$$

**Production (Eq. 14)**

$$
x_{i,t} = \min\{x^{\text{cap}}_{i,t},\; x^{\text{inp}}_{i,t},\; d_{i,t}\}
$$

**Labour adjustment (Eq. 19--20)**

$$
l_{i,t} = l_{i,t-1} + \gamma\,\frac{l_{i,0}}{x_{i,0}}
  \left[\min\{x^{\text{inp}}_{i,t},\, d_{i,t}\} - x^{\text{cap}}_{i,t}\right]
$$

See the Equations page for the complete set of behavioral equations.

## API Example

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

# Ensure src is on path when running the notebook directly
root = os.path.abspath(os.getcwd())
src_path = os.path.join(root, "src")
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

from macrostat.models.PichlerEtAl2022DIO import (
    PichlerEtAl2022DIO,
    ParametersPichlerEtAl2022DIO,
    VariablesPichlerEtAl2022DIO,
    ScenariosPichlerEtAl2022DIO,
)

_to_np = lambda x: x.detach().cpu().squeeze().numpy() if hasattr(x, "detach") else np.asarray(x)

### Loading Data

The model requires WIOD UK data, IHS criticality survey results, and
ONS inventory data.  If the replication-code directory is available,
set the ``PICHLER2022_DATA_DIR`` environment variable to its root path.
Otherwise the cells below will use default (zero) parameters and the
simulation will run in steady state.

In [ ]:
DATA_ROOT = os.environ.get("PICHLER2022_DATA_DIR", "")
HAS_DATA = bool(DATA_ROOT) and os.path.isdir(DATA_ROOT)

if HAS_DATA:
    from pathlib import Path
    data_dir = Path(DATA_ROOT) / "data" / "uk"
    ihs_dir = Path(DATA_ROOT) / "data" / "ihs"
    inv_file = Path(DATA_ROOT) / "data" / "uk" / "ons_table_ratio_inv_go.csv"
    shock_csv = Path(DATA_ROOT) / "data" / "shocks" / "shock_scenarios.csv"

    params = ParametersPichlerEtAl2022DIO.from_wiod_uk(
        data_dir=data_dir,
        ihs_dir=ihs_dir,
        inv_file=inv_file,
    )
    print(f"Loaded WIOD UK data from {DATA_ROOT}")
else:
    params = ParametersPichlerEtAl2022DIO()
    print("No data directory found; using default (zero) parameters.")

variables = VariablesPichlerEtAl2022DIO(parameters=params)
scenarios = ScenariosPichlerEtAl2022DIO(parameters=params)
model = PichlerEtAl2022DIO(parameters=params, variables=variables, scenarios=scenarios)

## Baseline Simulation (No Shocks)

Running the model without any shocks should maintain the initial
steady state across all 182 days.

In [ ]:
model.simulate()
ts = model.variables.timeseries

x = _to_np(ts["GrossOutput"])
T = x.shape[0]

if x.ndim == 2:
    agg_output = x.sum(axis=1)
else:
    agg_output = x

if HAS_DATA and agg_output[0] > 0:
    normalised = agg_output / agg_output[0]
    print(f"Aggregate output range: {normalised.min():.6f} -- {normalised.max():.6f}")
    print("(Should be ~1.0 throughout if steady state is maintained.)")
else:
    print(f"Aggregate output (raw): min={agg_output.min():.4f}, max={agg_output.max():.4f}")

In [ ]:
if HAVE_PLT and HAS_DATA and agg_output[0] > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(normalised, color="k", linewidth=2)
    ax.set_xlabel("Day")
    ax.set_ylabel("Aggregate Output (normalised)")
    ax.set_title("Baseline: Steady-State Maintenance")
    ax.set_ylim(0.95, 1.05)
    plt.tight_layout()
    plt.show()

## Pandemic Shock Scenario

If data is available, we load the UK pandemic scenario from the
shock CSV and simulate the model with the half-critical production
function (the paper's baseline).

In [ ]:
if HAS_DATA:
    scenarios_shock = ScenariosPichlerEtAl2022DIO.from_shocks_csv(
        parameters=params,
        shock_csv=shock_csv,
    )
    model_shock = PichlerEtAl2022DIO(
        parameters=params,
        variables=VariablesPichlerEtAl2022DIO(parameters=params),
        scenarios=scenarios_shock,
    )
    model_shock.simulate(scenario=0)
    ts_shock = model_shock.variables.timeseries

    x_shock = _to_np(ts_shock["GrossOutput"])
    agg_shock = x_shock.sum(axis=1) if x_shock.ndim == 2 else x_shock
    agg_shock_norm = agg_shock / agg_output[0]
    print(f"Min aggregate output (shock): {agg_shock_norm.min():.4f}")
else:
    print("Skipping shock scenario (no data available).")

In [ ]:
if HAVE_PLT and HAS_DATA and agg_output[0] > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(normalised, color="grey", linewidth=1, label="Baseline")
    ax.plot(agg_shock_norm, color="tab:red", linewidth=2, label="Pandemic shock")
    ax.set_xlabel("Day")
    ax.set_ylabel("Aggregate Output (normalised)")
    ax.set_title("Half-Critical PBL: UK Pandemic Shock")
    ax.legend()
    plt.tight_layout()
    plt.show()

### Sectoral Output

The model tracks output for each of the 55 WIOD industries.  Below we
plot sectoral output normalised to pre-lockdown levels, coloured by
broad sector group.

In [ ]:
if HAVE_PLT and HAS_DATA and x_shock.ndim == 2:
    x0 = _to_np(params["InitialGrossOutput"])
    safe_x0 = np.where(x0 > 0, x0, 1.0)
    x_norm = x_shock / safe_x0[None, :]

    fig, ax = plt.subplots(figsize=(10, 5))
    for j in range(x_norm.shape[1]):
        ax.plot(x_norm[:, j], linewidth=0.7, alpha=0.5)
    ax.plot(agg_shock_norm, color="k", linewidth=2.5, label="Aggregate")
    ax.set_xlabel("Day")
    ax.set_ylabel("Output (normalised to pre-lockdown)")
    ax.set_title("Sectoral Gross Output under Pandemic Shock")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Alternative Production Function

The model supports five production function specifications.  Below we
compare the Leontief production function (most restrictive) with the
half-critical baseline.

In [ ]:
if HAS_DATA:
    params_leon = ParametersPichlerEtAl2022DIO.from_wiod_uk(
        data_dir=data_dir,
        ihs_dir=ihs_dir,
        inv_file=inv_file,
        hyperparameters={"production_function": "leontief"},
    )
    scenarios_leon = ScenariosPichlerEtAl2022DIO.from_shocks_csv(
        parameters=params_leon,
        shock_csv=shock_csv,
    )
    model_leon = PichlerEtAl2022DIO(
        parameters=params_leon,
        variables=VariablesPichlerEtAl2022DIO(parameters=params_leon),
        scenarios=scenarios_leon,
    )
    model_leon.simulate(scenario=0)
    ts_leon = model_leon.variables.timeseries

    x_leon = _to_np(ts_leon["GrossOutput"])
    agg_leon = x_leon.sum(axis=1) if x_leon.ndim == 2 else x_leon
    agg_leon_norm = agg_leon / agg_output[0]
    print(f"Min aggregate output (Leontief): {agg_leon_norm.min():.4f}")
else:
    print("Skipping alternative production function (no data available).")

In [ ]:
if HAVE_PLT and HAS_DATA and agg_output[0] > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(agg_shock_norm, color="tab:red", linewidth=2, label="Half-critical (baseline)")
    ax.plot(agg_leon_norm, color="tab:blue", linewidth=2, label="Leontief")
    ax.plot(normalised, color="grey", linewidth=1, linestyle="--", label="No shock")
    ax.set_xlabel("Day")
    ax.set_ylabel("Aggregate Output (normalised)")
    ax.set_title("Production Function Comparison under Pandemic Shock")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Source

Pichler, A., Pangallo, M., del Rio-Chanona, R.M., Lafond, F. & Farmer, J.D. (2022).
Forecasting the propagation of pandemic shocks with a dynamic input-output model.
*Journal of Economic Dynamics and Control*, 144, 104527.
[https://doi.org/10.1016/j.jedc.2022.104527](https://doi.org/10.1016/j.jedc.2022.104527)